# Bias-Variance Tradeoff in Machine Learning

This notebook explores the fundamental concept of the bias-variance tradeoff in machine learning, which is critical for building models that generalize well to unseen data.

## Learning Objectives
- Understand the concepts of bias and variance
- Visualize the bias-variance tradeoff
- Demonstrate the impact of model complexity on bias and variance
- Apply techniques to find the optimal balance between bias and variance

## 1. Import Required Libraries

We'll need various libraries for data manipulation, visualization, and building machine learning models.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_regression

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16

# Set random seed for reproducibility
np.random.seed(42)

## 2. Understanding Bias and Variance

### Key Definitions:

**Bias**: The error from erroneous assumptions in the learning algorithm. High bias can cause an algorithm to miss relevant relations between features and target outputs (underfitting).

**Variance**: The error from sensitivity to small fluctuations in the training set. High variance can cause an algorithm to model random noise in the training data rather than the intended outputs (overfitting).

### The Tradeoff:

The bias-variance tradeoff is the balance between:
- Models that are too simple (high bias, low variance)
- Models that are too complex (low bias, high variance)

Ideally, we want a model with low bias and low variance, but this is typically impossible to achieve simultaneously.

In [ ]:
# Create a simple visualization of the bias-variance concepts
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# High Bias (Underfitting) visualization
x = np.linspace(0, 10, 100)
y_true = np.sin(x) + np.random.normal(0, 0.2, 100)
x_plot = np.linspace(0, 10, 1000)
y_pred_biased = 0.5 * x_plot + 2  # Simple linear model

ax[0].scatter(x, y_true, alpha=0.5, label='Data points')
ax[0].plot(x_plot, np.sin(x_plot), 'g-', linewidth=2, label='True function')
ax[0].plot(x_plot, y_pred_biased, 'r-', linewidth=3, label='High bias model')
ax[0].set_title('High Bias (Underfitting)')
ax[0].legend()

# High Variance (Overfitting) visualization
poly_features = PolynomialFeatures(degree=15)
x_poly = poly_features.fit_transform(x.reshape(-1, 1))
model = LinearRegression()
model.fit(x_poly, y_true)

x_plot_poly = poly_features.transform(x_plot.reshape(-1, 1))
y_pred_overfit = model.predict(x_plot_poly)

ax[1].scatter(x, y_true, alpha=0.5, label='Data points')
ax[1].plot(x_plot, np.sin(x_plot), 'g-', linewidth=2, label='True function')
ax[1].plot(x_plot, y_pred_overfit, 'r-', linewidth=3, label='High variance model')
ax[1].set_title('High Variance (Overfitting)')
ax[1].legend()

plt.tight_layout()
plt.show()

## 3. Mathematical Decomposition of Error

The expected prediction error for any machine learning algorithm can be broken down into three parts:

### 1. Bias
The bias term represents how much the average model prediction differs from the true value we're trying to predict.

### 2. Variance
The variance term represents how much the model prediction changes when using different training sets.

### 3. Irreducible Error
The noise term represents random variation in the target variable that cannot be eliminated by any model.

### The Equation
For a given data point x, the expected mean squared error can be decomposed as:

$$\text{Expected Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

Or more formally:

$$E[(y - \hat{f}(x))^2] = [E[\hat{f}(x)] - f(x)]^2 + E[(\hat{f}(x) - E[\hat{f}(x)])^2] + \sigma^2$$

Where:
- $f(x)$ is the true function
- $\hat{f}(x)$ is our model's prediction
- $\sigma^2$ is the irreducible error (noise)

## 4. Visualizing the Bias-Variance Tradeoff

Let's create a visualization to demonstrate how bias and variance change with model complexity.

In [ ]:
# Function to generate a dataset with a non-linear relationship
def generate_nonlinear_data(n_samples=100, noise=0.3):
    X = np.random.uniform(0, 10, size=n_samples)
    y = np.sin(X) + noise * np.random.randn(n_samples)
    return X.reshape(-1, 1), y

# Function to estimate bias and variance using multiple training sets
def bias_variance_decomposition(degrees, n_repeats=100, test_size=200):
    # Generate a large test set
    X_test = np.linspace(0, 10, test_size).reshape(-1, 1)
    y_test = np.sin(X_test.ravel())  # True function without noise
    
    avg_bias_squared = np.zeros(len(degrees))
    avg_var = np.zeros(len(degrees))
    avg_mse = np.zeros(len(degrees))
    
    for i, degree in enumerate(degrees):
        # Store predictions from different models
        predictions = np.zeros((n_repeats, test_size))
        
        for j in range(n_repeats):
            # Generate a different training set each time
            X_train, y_train = generate_nonlinear_data(n_samples=50)
            
            # Train a polynomial model
            model = make_pipeline(
                PolynomialFeatures(degree=degree),
                LinearRegression()
            )
            model.fit(X_train, y_train)
            
            # Predict on test set
            predictions[j] = model.predict(X_test).ravel()
            
        # Calculate average prediction across all models
        avg_prediction = np.mean(predictions, axis=0)
        
        # Calculate squared bias: (avg_prediction - y_test)^2
        bias_squared = np.mean((avg_prediction - y_test) ** 2)
        
        # Calculate variance: average of (prediction - avg_prediction)^2
        variance = np.mean(np.var(predictions, axis=0))
        
        # Calculate average MSE
        mse = np.mean(np.mean((predictions - y_test.reshape(1, -1)) ** 2, axis=1))
        
        avg_bias_squared[i] = bias_squared
        avg_var[i] = variance
        avg_mse[i] = mse
    
    return avg_bias_squared, avg_var, avg_mse

# Calculate bias-variance decomposition for different polynomial degrees
degrees = [1, 2, 3, 4, 5, 7, 9, 11, 13, 15, 17, 19]
bias_squared, variance, total_error = bias_variance_decomposition(degrees)

# Plot the results
plt.figure(figsize=(12, 7))
plt.plot(degrees, bias_squared, 'o-', color='blue', label='Bias²')
plt.plot(degrees, variance, 'o-', color='orange', label='Variance')
plt.plot(degrees, total_error, 'o-', color='red', label='Total Error')
plt.plot(degrees, bias_squared + variance, 'o-', color='purple', linestyle='--', 
         label='Bias² + Variance')

plt.xlabel('Model Complexity (Polynomial Degree)')
plt.ylabel('Error')
plt.title('Bias-Variance Tradeoff with Increasing Model Complexity')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Practical Examples with Models

Let's implement different polynomial regression models with varying degrees to demonstrate underfitting, good fit, and overfitting on a simple dataset.

In [ ]:
# Generate a dataset for demonstration
np.random.seed(42)
X = np.sort(5 * np.random.rand(80, 1), axis=0)
y = np.sin(X).ravel() + np.random.normal(0, 0.1, size=X.shape[0])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create test points for prediction
X_plot = np.linspace(0, 5, 1000).reshape(-1, 1)

# Define polynomial degrees to try
degrees = [1, 3, 10]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, degree in enumerate(degrees):
    # Create and fit the polynomial regression model
    poly_model = make_pipeline(
        PolynomialFeatures(degree=degree),
        LinearRegression()
    )
    poly_model.fit(X_train, y_train)
    
    # Predict on test data and calculate error
    y_train_pred = poly_model.predict(X_train)
    y_test_pred = poly_model.predict(X_test)
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    
    # Make predictions on the plot points
    y_plot = poly_model.predict(X_plot)
    
    # Plot the results
    axes[i].scatter(X_train, y_train, color='blue', alpha=0.6, label='Training data')
    axes[i].scatter(X_test, y_test, color='green', alpha=0.6, label='Test data')
    axes[i].plot(X_plot, y_plot, color='red', label=f'Polynomial degree {degree}')
    axes[i].plot(X_plot, np.sin(X_plot), color='black', linestyle='--', label='True function')
    axes[i].set_ylim(-1.5, 1.5)
    axes[i].set_title(f'Degree {degree} Polynomial\nTrain MSE: {train_mse:.4f}, Test MSE: {test_mse:.4f}')
    axes[i].legend()
    
    if i == 0:
        title = "Underfitting (High Bias)"
    elif i == 1:
        title = "Good Balance"
    else:
        title = "Overfitting (High Variance)"
    
    axes[i].set_xlabel('X')
    axes[i].set_ylabel('y')
    axes[i].text(0.05, -1.2, title, fontsize=14, bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

## 6. Model Complexity and the Bias-Variance Tradeoff

Different models have inherently different bias-variance characteristics. Let's compare several common models and see how they perform on the same dataset.

In [ ]:
# Create a slightly more complex dataset
np.random.seed(0)
X, y = make_regression(n_samples=100, n_features=1, noise=20, random_state=42)
X = np.sort(X, axis=0)
y = np.sin(X.ravel() * 3) * 10 + X.ravel() * 5 + np.random.randn(100) * 5

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create test points for smooth visualization
X_plot = np.linspace(X.min(), X.max(), 1000).reshape(-1, 1)

# Define models with increasing complexity
models = [
    ("Linear Regression", LinearRegression()),
    ("Polynomial Regression (degree=2)", make_pipeline(PolynomialFeatures(2), LinearRegression())),
    ("Polynomial Regression (degree=5)", make_pipeline(PolynomialFeatures(5), LinearRegression())),
    ("Decision Tree (max_depth=2)", DecisionTreeRegressor(max_depth=2)),
    ("Decision Tree (max_depth=5)", DecisionTreeRegressor(max_depth=5)),
    ("Decision Tree (unlimited)", DecisionTreeRegressor())
]

# Create subplots - 2 rows, 3 columns
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Fit each model and plot results
for i, (name, model) in enumerate(models):
    model.fit(X_train, y_train)
    
    # Calculate train and test errors
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    
    # Make predictions for plotting
    try:
        y_plot = model.predict(X_plot)
    except:
        # Some models might need reshaping
        y_plot = model.predict(X_plot.reshape(-1, 1))
    
    # Plot the results
    axes[i].scatter(X_train, y_train, color='blue', alpha=0.5, label='Training data')
    axes[i].scatter(X_test, y_test, color='green', alpha=0.5, label='Test data')
    axes[i].plot(X_plot, y_plot, color='red', linewidth=2, label='Model prediction')
    axes[i].set_title(f"{name}\nTrain MSE: {train_mse:.2f}, Test MSE: {test_mse:.2f}")
    axes[i].set_xlabel('X')
    axes[i].set_ylabel('y')
    axes[i].legend(loc='best')
    
    # Add text describing bias-variance characteristics
    if "Linear" in name:
        bias_var_text = "Higher Bias, Lower Variance"
    elif "degree=2" in name:
        bias_var_text = "Medium Bias, Medium Variance"
    elif "degree=5" in name:
        bias_var_text = "Lower Bias, Higher Variance"
    elif "max_depth=2" in name:
        bias_var_text = "Medium Bias, Medium Variance"
    elif "max_depth=5" in name:
        bias_var_text = "Lower Bias, Higher Variance"
    else:
        bias_var_text = "Very Low Bias, Very High Variance"
        
    axes[i].text(X.min() + 0.1, np.max(y) - 10, bias_var_text, 
                 bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

## 7. Regularization Techniques

Regularization methods like Ridge (L2) and Lasso (L1) regression help manage the bias-variance tradeoff by imposing penalties on model complexity. Let's see how these techniques work in practice.

In [ ]:
# Create a dataset where regularization will be helpful
np.random.seed(42)
n_samples, n_features = 100, 20
X = np.random.randn(n_samples, n_features)

# True model: only the first 5 features matter, others are noise
beta = np.zeros(n_features)
beta[:5] = np.array([5, 4.5, 4, 3.5, 3])
y = X.dot(beta) + np.random.randn(n_samples) * 2

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define regularization strengths to test
alphas = [0, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

# Compare Linear Regression, Ridge, and Lasso
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(max_iter=10000)
}

# Plot setups
fig, axes = plt.subplots(len(models), 1, figsize=(12, 12), sharex=True)

# For storing results
results = np.zeros((len(models), len(alphas), 2))  # [model_idx, alpha_idx, (train_mse, test_mse)]
coefficients = np.zeros((len(models), len(alphas), n_features))

# Train and evaluate each model with different regularization strengths
for model_idx, (model_name, model_class) in enumerate(models.items()):
    train_errors = []
    test_errors = []
    
    for alpha_idx, alpha in enumerate(alphas):
        # Skip alpha for Linear Regression
        if model_name == 'Linear Regression' and alpha > 0:
            # Store the same values for different alphas in Linear Regression
            results[model_idx, alpha_idx, 0] = results[model_idx, 0, 0]
            results[model_idx, alpha_idx, 1] = results[model_idx, 0, 1]
            coefficients[model_idx, alpha_idx] = coefficients[model_idx, 0]
            continue
            
        # Create model with current alpha
        if model_name == 'Linear Regression':
            model = model_class
        else:
            model = model_class(alpha=alpha)
            
        # Train model
        model.fit(X_train, y_train)
        
        # Calculate errors
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        train_mse = mean_squared_error(y_train, train_pred)
        test_mse = mean_squared_error(y_test, test_pred)
        
        # Store results
        results[model_idx, alpha_idx, 0] = train_mse
        results[model_idx, alpha_idx, 1] = test_mse
        
        # Store coefficients
        if hasattr(model, 'coef_'):
            coefficients[model_idx, alpha_idx] = model.coef_
        else:
            coefficients[model_idx, alpha_idx] = np.zeros(n_features)

# Plot results
for model_idx, model_name in enumerate(models.keys()):
    ax = axes[model_idx]
    
    # Plot training and test errors
    ax.semilogx(alphas, results[model_idx, :, 0], 'b--o', label='Training MSE')
    ax.semilogx(alphas, results[model_idx, :, 1], 'r-^', label='Test MSE')
    ax.set_ylabel('Mean Squared Error')
    ax.set_title(f'{model_name}')
    ax.legend()
    
    # Add grid
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    
    # For Linear Regression, only the first value is relevant
    if model_name == 'Linear Regression':
        ax.text(0.5, 0.5, 'Linear Regression has no regularization parameter', 
                horizontalalignment='center', verticalalignment='center', 
                transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.8))

axes[-1].set_xlabel('Regularization Strength (α)')
plt.tight_layout()
plt.show()

# Plot coefficients
plt.figure(figsize=(15, 10))
for model_idx, model_name in enumerate(models.keys()):
    if model_name == 'Linear Regression':
        # For Linear Regression, just plot the coefficients once
        plt.plot(range(n_features), coefficients[model_idx, 0], 'o-', 
                 label=f'{model_name}', linewidth=2)
    else:
        # For Ridge and Lasso, plot coefficients with different alphas
        for alpha_idx, alpha in enumerate(alphas):
            if alpha_idx in [0, 2, 4, 6]:  # Select a subset of alphas for clarity
                plt.plot(range(n_features), coefficients[model_idx, alpha_idx], 'o-', 
                         label=f'{model_name}, α={alpha}', alpha=0.7)

# Plot the true coefficients
plt.plot(range(n_features), beta, 'k*--', markersize=10, label='True Coefficients', linewidth=2)

plt.title('Model Coefficients with Different Regularization Strengths')
plt.xlabel('Feature Index')
plt.ylabel('Coefficient Value')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 8. Cross-Validation to Find Optimal Model Complexity

Cross-validation helps us objectively determine the optimal model complexity that balances bias and variance. Let's use it to find the best polynomial degree for our regression task.

In [ ]:
# Create a dataset for polynomial regression
np.random.seed(42)
X = np.sort(np.random.uniform(0, 1, 50)).reshape(-1, 1)
y = np.sin(2 * np.pi * X).ravel() + np.random.normal(0, 0.2, X.shape[0])

# Define polynomial degrees to evaluate
degrees = range(1, 15)

# Perform cross-validation for each degree
train_scores = []
cv_scores = []

for degree in degrees:
    poly_model = make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression()
    )
    
    # Compute cross-validation scores
    cv_score = -np.mean(cross_val_score(poly_model, X, y, cv=5, 
                                        scoring='neg_mean_squared_error'))
    cv_scores.append(cv_score)
    
    # Compute training scores
    poly_model.fit(X, y)
    train_score = mean_squared_error(y, poly_model.predict(X))
    train_scores.append(train_score)

# Find the best degree
best_degree = degrees[np.argmin(cv_scores)]

# Plot the cross-validation and training errors
plt.figure(figsize=(10, 6))
plt.plot(degrees, train_scores, 'o-', color='blue', label='Training MSE')
plt.plot(degrees, cv_scores, 'o-', color='red', label='Cross-validation MSE')
plt.axvline(best_degree, color='green', linestyle='--', label=f'Best degree: {best_degree}')
plt.xlabel('Polynomial Degree')
plt.ylabel('Mean Squared Error')
plt.title('Model Selection with Cross-Validation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Fit the best model
best_model = make_pipeline(
    PolynomialFeatures(best_degree),
    LinearRegression()
)
best_model.fit(X, y)

# Create a more detailed plot with the best model
X_plot = np.linspace(0, 1, 1000).reshape(-1, 1)
y_plot = best_model.predict(X_plot)
y_true = np.sin(2 * np.pi * X_plot).ravel()

# Also plot some suboptimal models for comparison
models_to_compare = [1, best_degree, 14]  # Underfitting, best, overfitting
plt.figure(figsize=(12, 6))

plt.scatter(X, y, color='black', label='Data points', alpha=0.6)
plt.plot(X_plot, y_true, color='green', label='True function', linewidth=2)

for degree in models_to_compare:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X, y)
    y_model = model.predict(X_plot)
    
    if degree == 1:
        label = f'Degree {degree} (High bias)'
        color = 'blue'
    elif degree == best_degree:
        label = f'Degree {degree} (Best balance)'
        color = 'red'
    else:
        label = f'Degree {degree} (High variance)'
        color = 'orange'
    
    plt.plot(X_plot, y_model, color=color, label=label, alpha=0.7, linewidth=2)

plt.xlabel('X')
plt.ylabel('y')
plt.title('Polynomial Models with Different Degrees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Real-world Dataset Demonstration

Let's apply what we've learned about the bias-variance tradeoff to a real-world dataset: the Boston Housing dataset. We'll analyze the tradeoff using learning curves and different model complexities.

In [ ]:
# Import the Boston Housing dataset
from sklearn.datasets import load_boston

# Load dataset
try:
    boston = load_boston()
    X_boston = boston.data
    y_boston = boston.target
    feature_names = boston.feature_names
except:
    # In newer versions of scikit-learn, load_boston may be deprecated
    # Alternative loading approach
    from sklearn.datasets import fetch_california_housing
    boston = fetch_california_housing()
    X_boston = boston.data
    y_boston = boston.target
    feature_names = boston.feature_names
    print("Using California Housing dataset instead of Boston")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_boston, y_boston, test_size=0.3, random_state=42
)

# Function to plot learning curves
def plot_learning_curve(estimator, title, X, y, ylim=None, cv=5,
                        n_jobs=-1, train_sizes=np.linspace(.1, 1.0, 10)):
    plt.figure(figsize=(10, 6))
    plt.title(title)
    
    if ylim is not None:
        plt.ylim(*ylim)
        
    plt.xlabel("Training examples")
    plt.ylabel("Mean Squared Error")
    
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes,
        scoring='neg_mean_squared_error')
    
    train_scores = -train_scores
    test_scores = -test_scores
    
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    
    plt.grid()
    
    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="b")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="r")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="b", label="Training score")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="r", label="Cross-validation score")
    
    plt.legend(loc="best")
    return plt

# Define models with different complexity
models = [
    ("Linear Regression (Simple)", LinearRegression()),
    ("Decision Tree (max_depth=2)", DecisionTreeRegressor(max_depth=2)),
    ("Decision Tree (max_depth=5)", DecisionTreeRegressor(max_depth=5)),
    ("Decision Tree (unlimited)", DecisionTreeRegressor()),
    ("Ridge Regression (alpha=1.0)", Ridge(alpha=1.0)),
    ("Ridge Regression (alpha=0.1)", Ridge(alpha=0.1))
]

# Plot learning curves for each model
for name, model in models:
    plot_learning_curve(
        model, 
        f"Learning Curve for {name}",
        X_train, y_train, ylim=(0, 100)
    )
    plt.tight_layout()
    plt.show()

# Compare final performance on test set
results = []

for name, model in models:
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    
    results.append({
        'Model': name,
        'Train MSE': train_mse,
        'Test MSE': test_mse,
        'Gap': train_mse - test_mse
    })

# Convert results to DataFrame for nice display
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test MSE')

# Plot comparison
plt.figure(figsize=(12, 6))
bar_width = 0.35
index = np.arange(len(models))

plt.barh(index, results_df['Train MSE'], bar_width, color='blue', alpha=0.7, label='Train MSE')
plt.barh(index + bar_width, results_df['Test MSE'], bar_width, color='red', alpha=0.7, label='Test MSE')

plt.ylabel('Model')
plt.xlabel('Mean Squared Error')
plt.title('Model Performance Comparison (Lower is Better)')
plt.yticks(index + bar_width/2, results_df['Model'])
plt.legend()
plt.tight_layout()
plt.show()

# Print results table
print("Model Performance Summary:")
print(results_df.round(2))

## Conclusion: Finding the Sweet Spot

The bias-variance tradeoff is fundamental to building effective machine learning models. Here's what we've learned:

1. **Understanding the Tradeoff:**
   - **High bias (underfitting)**: Model is too simple, misses important patterns
   - **High variance (overfitting)**: Model is too complex, captures noise

2. **Mathematical decomposition** of the prediction error into bias, variance, and irreducible error components helps us understand model behavior.

3. **Model complexity** directly impacts the bias-variance balance:
   - Simple models tend to have high bias and low variance
   - Complex models tend to have low bias and high variance

4. **Regularization techniques** like Ridge and Lasso regression help manage the tradeoff by penalizing model complexity.

5. **Cross-validation** is critical for finding the optimal model complexity that balances bias and variance.

6. The **learning curves** help diagnose whether a model is suffering from high bias or high variance:
   - Large gap between training and validation performance indicates high variance
   - Poor performance on both training and validation indicates high bias

Finding the right balance between bias and variance is essential for creating models that generalize well to new, unseen data.